# **KNN Imputer**

* Works on algorithm : KNN
* If there is a value missing in a row so according it we would try to fill it with some value of the row that matches the most.
* This similarity is calculated using the euclidean distances. under root   [(x1-x2)^2 + (y1 -y2)^2 + ...], we find euclidean distance with all the other rows nd choose one with smallest value.
* But the problem here lies in there might be some values that are missing, there for we use a revised formaula **nan euclidean distance** :  [weight {(x1-x2)^2 + (y1 -y2)^2 + ...}], where weight is total values divide by not null values.

**Advantages:**
* More accurate than cca etc.

**Disadvantages:**
* More no of calculations
* For deployment on server, have to provide whole training set to the server -> inc energy consumption.

In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [15]:
df = pd.read_csv('/content/train.csv')[['Age','Pclass','Fare','Survived']]

In [16]:
df.head()

,Age,Pclass,Fare,Survived
0,22.0,3,7.2500,0
1,38.0,1,71.2833,1
2,26.0,3,7.9250,1
3,35.0,1,53.1000,1
4,35.0,3,8.0500,0


In [17]:
df.isnull().mean() * 100

,0
Age,19.86532
Pclass,0.00000
Fare,0.00000
Survived,0.00000


In [18]:
X= df.drop(columns=['Survived'])
y = df['Survived']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [20]:
X_train.head()

,Age,Pclass,Fare
30,40.0,1,27.7208
10,4.0,3,16.7000
873,47.0,3,9.0000
182,9.0,3,31.3875
876,20.0,3,9.8458


In [21]:
knn = KNNImputer() #also try its various parameter for better result.

X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)

In [22]:
#Applying algo:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred)

0.7039106145251397

In [23]:
#Comparision with simpleImputer -> mean
si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [24]:
#Applying algo:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred2 = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.7039106145251397

# **Iterative Imputer/ Mice:**
* MICE - Multivariate Imputation by Chained Equations.
* Can only work when :
  -> MAR(missing at random) : like optionally data not filled, such as rating etc. In MAR by using other columns we can predict the missing values.

**Advantage:**
* Data is more accurate.

**Disadvantage**
* slow processing, bcz of mar processing.
* training data on server.

* Mice is always implmented on input columns.
*

In [25]:
dt = pd.read_csv('/content/50_Startups.csv')

In [26]:
dt.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [27]:
dt.isnull().mean()*100

,0
R&D Spend,0.0
Administration,0.0
Marketing Spend,0.0
State,0.0
Profit,0.0


### Step-01 :
* Replace nan value with mean of respective columns.
* Again replace that mean with nan, while doing this we apply algorithm like logisticReg etc and the column serves as target while other input serves as predictor to predict that nan value.
* Apply this for all columns having missing values.
* Iteration zer was when we replace nan with mean, iteration one was when we replace value with predicted value.
* In the next step, we subtract It1 - IT0, therefore all value becomes zero except predicted one, we keep repeating this process until these predicted values becomes zero or near to zero.


In [28]:
# refer to https://github.com/campusx-official/100-days-of-machine-learning/blob/main/day40-iterative-imputer/50_Startups.csv    for code